# Catch Me If You Can Network Analysis (Weeks 1-3)

**Network:** *Catch Me If You Can* movie network  
**Type:** undirected, weighted social network  
**Meaning of nodes:** characters  
**Meaning of edges:** two characters appear in the same scene  
**Meaning of weights:** number of shared-scene appearances

**Source:** J. Kaminski et al., *Moviegalaxies - Social Networks in Movies*, Harvard Dataverse, V3 (2018).

## Week 1

In this first part, we load the network, represent it as a Python graph, draw it, and compute the basic descriptive measures required by the assignment: number of nodes, number of edges, average degree, and density.

In [ ]:
from pathlib import Path
from itertools import combinations

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")

DATA_DIR = Path.cwd() / "datasets"
NODES_PATH = DATA_DIR / "nodes.csv"
EDGES_PATH = DATA_DIR / "edges.csv"

nodes_df = pd.read_csv(NODES_PATH)
edges_df = pd.read_csv(EDGES_PATH)

nodes_df.columns = [col.strip() for col in nodes_df.columns]
edges_df.columns = [col.strip() for col in edges_df.columns]

G = nx.Graph()

for _, row in nodes_df.iterrows():
    G.add_node(int(row["# index"]), label=str(row["label"]).strip())

for _, row in edges_df.iterrows():
    G.add_edge(int(row["# source"]), int(row["target"]), weight=float(row["weight"]))

print(f"Nodes loaded: {G.number_of_nodes()}")
print(f"Edges loaded: {G.number_of_edges()}")

nodes_df.head(), edges_df.head()

### Graph visualization

The dataset available in this folder does not include predefined plotting coordinates, so the graph is drawn with a force-directed layout (`spring_layout`). This still provides a valid visual overview of the network structure.

In [ ]:
plt.figure(figsize=(14, 10))
layout = nx.spring_layout(G, seed=42, k=0.45)
weighted_degree = dict(G.degree(weight='weight'))
node_sizes = [80 + 35 * weighted_degree[node] for node in G.nodes()]

nx.draw_networkx_edges(G, pos=layout, alpha=0.35, edge_color='gray')
nx.draw_networkx_nodes(G, pos=layout, node_size=node_sizes, node_color='steelblue', alpha=0.9)
nx.draw_networkx_labels(
    G,
    pos=layout,
    labels={node: data['label'] for node, data in G.nodes(data=True)},
    font_size=8,
)

plt.title('Catch Me If You Can character network')
plt.axis('off')
plt.show()

### Basic network statistics

In [ ]:
num_nodes = G.number_of_nodes()
num_edges = G.number_of_edges()
average_degree = sum(dict(G.degree()).values()) / num_nodes
density = nx.density(G)

summary_week1 = pd.DataFrame(
    {
        'metric': ['Number of nodes', 'Number of edges', 'Average degree', 'Density'],
        'value': [num_nodes, num_edges, average_degree, density],
    }
)

summary_week1

### Comment

The network contains a moderate number of characters but a relatively small number of realized ties compared with all possible ties. The average degree tells us that each character is connected, on average, to about four others through shared scenes. The density is low, which is typical for movie character networks: most characters interact only with a limited subset of the cast rather than with everyone.

## Week 2

The assignment now asks us to work on the **largest connected component** of the network. We implement:

1. a custom function computing the clustering coefficient for every node;
2. a custom function computing the average clustering coefficient;
3. a comparison with the built-in NetworkX measures for average clustering and transitivity.

Here clustering is computed on the simple undirected topology of the component. The original graph is already undirected.

In [ ]:
if nx.is_connected(G):
    G_lcc = G.copy()
else:
    largest_component_nodes = max(nx.connected_components(G), key=len)
    G_lcc = G.subgraph(largest_component_nodes).copy()

print(f'Largest connected component: {G_lcc.number_of_nodes()} nodes, {G_lcc.number_of_edges()} edges')

In [ ]:
def local_clustering_manual(graph):
    clustering = {}

    for node in graph.nodes():
        neighbors = list(graph.neighbors(node))
        k = len(neighbors)

        if k < 2:
            clustering[node] = 0.0
            continue

        closed_triplets = 0
        for u, v in combinations(neighbors, 2):
            if graph.has_edge(u, v):
                closed_triplets += 1

        clustering[node] = closed_triplets / (k * (k - 1) / 2)

    return clustering


def average_clustering_manual(graph):
    clustering_values = local_clustering_manual(graph)
    return sum(clustering_values.values()) / len(clustering_values)


manual_clustering = local_clustering_manual(G_lcc)
manual_average_clustering = average_clustering_manual(G_lcc)
networkx_average_clustering = nx.average_clustering(G_lcc)
networkx_transitivity = nx.transitivity(G_lcc)

comparison_week2 = pd.DataFrame(
    {
        'measure': [
            'Manual average clustering',
            'NetworkX average clustering',
            'NetworkX transitivity',
        ],
        'value': [
            manual_average_clustering,
            networkx_average_clustering,
            networkx_transitivity,
        ],
    }
)

comparison_week2

In [ ]:
clustering_df = pd.DataFrame(
    {
        'node': list(manual_clustering.keys()),
        'label': [G_lcc.nodes[node]['label'] for node in manual_clustering.keys()],
        'clustering': list(manual_clustering.values()),
    }
).sort_values('clustering', ascending=False)

clustering_df.head(10)

### Comment

The manual average clustering should match the built-in `nx.average_clustering()` up to small numerical precision differences. This is a good consistency check for the custom implementation. The transitivity is related but not identical: it is a global triangle-based measure, while average clustering is the mean of local node-level coefficients. If the two values differ, it means that triangle closure is not distributed uniformly across nodes.

## Week 3

In this part we study how clustering is distributed across the largest connected component.

1. We compute the cumulative distribution of node clustering coefficients.
2. For each node, we compute the average clustering coefficient of its neighbors.
3. We compute the cumulative distribution of this second quantity.
4. We compare the two distributions.

In [ ]:
def empirical_cdf(values):
    values = np.sort(np.asarray(values, dtype=float))
    cumulative = np.arange(1, len(values) + 1) / len(values)
    return values, cumulative


clustering_values = list(manual_clustering.values())
clustering_x, clustering_cdf = empirical_cdf(clustering_values)

In [ ]:
def average_neighbor_clustering(graph, clustering_by_node):
    neighbor_average = {}

    for node in graph.nodes():
        neighbors = list(graph.neighbors(node))
        if not neighbors:
            neighbor_average[node] = 0.0
            continue

        neighbor_average[node] = float(np.mean([clustering_by_node[neighbor] for neighbor in neighbors]))

    return neighbor_average


neighbor_clustering = average_neighbor_clustering(G_lcc, manual_clustering)
neighbor_values = list(neighbor_clustering.values())
neighbor_x, neighbor_cdf = empirical_cdf(neighbor_values)

neighbor_clustering_df = pd.DataFrame(
    {
        'node': list(neighbor_clustering.keys()),
        'label': [G_lcc.nodes[node]['label'] for node in neighbor_clustering.keys()],
        'avg_neighbor_clustering': list(neighbor_clustering.values()),
    }
).sort_values('avg_neighbor_clustering', ascending=False)

neighbor_clustering_df.head(10)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].step(clustering_x, clustering_cdf, where='post', color='darkorange')
axes[0].set_title('CDF of node clustering')
axes[0].set_xlabel('Clustering coefficient')
axes[0].set_ylabel('Cumulative probability')

axes[1].step(neighbor_x, neighbor_cdf, where='post', color='teal')
axes[1].set_title('CDF of average neighbor clustering')
axes[1].set_xlabel('Average clustering of neighbors')
axes[1].set_ylabel('Cumulative probability')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
plt.step(clustering_x, clustering_cdf, where='post', label='Node clustering', linewidth=2)
plt.step(neighbor_x, neighbor_cdf, where='post', label='Average neighbor clustering', linewidth=2)
plt.xlabel('Clustering value')
plt.ylabel('Cumulative probability')
plt.title('Comparison of the two cumulative distributions')
plt.legend()
plt.show()

### Comment

This comparison helps us understand whether nodes tend to be surrounded by neighbors that are more clustered, less clustered, or similarly clustered compared with the nodes themselves. If the cumulative distribution of average neighbor clustering is shifted to the right, it suggests that many nodes are connected to neighbors embedded in tighter local triangles. If the two curves are close, local clustering and neighborhood clustering are distributed in a similar way across the component.

## Week 4

No Week 4 task was included in the assignment text you provided. For this reason, the notebook moves directly from Week 3 to Week 5.

## Week 5

We now select two centrality notions that are meaningful for this movie network:

1. **Betweenness centrality**, because it highlights brokerage characters that connect different parts of the story.
2. **PageRank**, because it captures global importance by rewarding connections to already important characters.

Since the graph is weighted and edge weights represent stronger interaction when they are larger, we treat weights carefully:
- for betweenness, we convert weight into a distance using `distance = 1 / weight`;
- for PageRank, we directly use the weights as interaction strength.

In [ ]:
for u, v, data in G.edges(data=True):
    data['distance'] = 1.0 / data['weight']


def pagerank_manual(graph, alpha=0.85, max_iter=200, tol=1.0e-9):
    nodes = list(graph.nodes())
    n = len(nodes)
    rank = {node: 1.0 / n for node in nodes}
    weighted_degree = {
        node: sum(data.get('weight', 1.0) for _, _, data in graph.edges(node, data=True))
        for node in nodes
    }

    for _ in range(max_iter):
        new_rank = {node: (1.0 - alpha) / n for node in nodes}

        dangling_sum = sum(rank[node] for node in nodes if weighted_degree[node] == 0)
        dangling_contribution = alpha * dangling_sum / n
        for node in nodes:
            new_rank[node] += dangling_contribution

        for source in nodes:
            if weighted_degree[source] == 0:
                continue
            for target in graph.neighbors(source):
                weight = graph[source][target].get('weight', 1.0)
                new_rank[target] += alpha * rank[source] * (weight / weighted_degree[source])

        error = sum(abs(new_rank[node] - rank[node]) for node in nodes)
        rank = new_rank
        if error < tol:
            break

    return rank


betweenness_scores = nx.betweenness_centrality(G, weight='distance')
pagerank_scores = pagerank_manual(G)

In [ ]:
centrality_df = pd.DataFrame(
    {
        'node': list(G.nodes()),
        'label': [G.nodes[node]['label'] for node in G.nodes()],
        'betweenness': [betweenness_scores[node] for node in G.nodes()],
        'pagerank': [pagerank_scores[node] for node in G.nodes()],
    }
)

centrality_df['betweenness_rank'] = centrality_df['betweenness'].rank(ascending=False, method='min')
centrality_df['pagerank_rank'] = centrality_df['pagerank'].rank(ascending=False, method='min')
centrality_df['combined_rank_score'] = centrality_df['betweenness_rank'] + centrality_df['pagerank_rank']

centrality_df.sort_values('combined_rank_score').head(10)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bet_top = centrality_df.sort_values('betweenness', ascending=False).head(10)
axes[0].barh(bet_top['label'][::-1], bet_top['betweenness'][::-1], color='indianred')
axes[0].set_title('Top 10 nodes by betweenness')
axes[0].set_xlabel('Betweenness centrality')

pr_top = centrality_df.sort_values('pagerank', ascending=False).head(10)
axes[1].barh(pr_top['label'][::-1], pr_top['pagerank'][::-1], color='slateblue')
axes[1].set_title('Top 10 nodes by PageRank')
axes[1].set_xlabel('PageRank score')

plt.tight_layout()
plt.show()

In [ ]:
most_central_betweenness = centrality_df.sort_values('betweenness', ascending=False).iloc[0]
most_central_pagerank = centrality_df.sort_values('pagerank', ascending=False).iloc[0]

print(
    f"Highest betweenness: {most_central_betweenness['label']} "
    f"(node {int(most_central_betweenness['node'])}, score = {most_central_betweenness['betweenness']:.4f})"
)
print(
    f"Highest PageRank: {most_central_pagerank['label']} "
    f"(node {int(most_central_pagerank['node'])}, score = {most_central_pagerank['pagerank']:.4f})"
)

### Comment

The same character appearing at the top of both rankings is strong evidence of structural importance. Betweenness emphasizes brokerage, so it identifies characters that sit on many shortest paths between others. PageRank emphasizes prestige, so it favors characters connected to already influential nodes. If one node dominates both measures, that node is central both as a bridge and as a globally important actor in the interaction network.

## Week 6

For community detection, the assignment requires us to treat the graph as **undirected and unweighted**, remove self-loops, and work on the **largest connected component**.

We implement two techniques:

1. **Bridge removal** through the Girvan-Newman algorithm, selecting the partition with the highest modularity.
2. **Modularity optimization** through greedy modularity maximization.

We then compare the resulting partitions and export the preferred one for visualization in Gephi Lite.

In [ ]:
G_week6 = nx.Graph()
G_week6.add_nodes_from(G.nodes(data=True))
G_week6.add_edges_from((u, v) for u, v in G.edges() if u != v)

if nx.is_connected(G_week6):
    G_week6_lcc = G_week6.copy()
else:
    week6_component_nodes = max(nx.connected_components(G_week6), key=len)
    G_week6_lcc = G_week6.subgraph(week6_component_nodes).copy()

print(
    f"Week 6 graph: {G_week6_lcc.number_of_nodes()} nodes, "
    f"{G_week6_lcc.number_of_edges()} edges"
)

In [ ]:
def best_girvan_newman_partition(graph):
    best_partition = None
    best_modularity = float('-inf')
    modularity_trace = []

    for partition in nx.community.girvan_newman(graph):
        partition = tuple(frozenset(community) for community in partition)
        modularity = nx.community.modularity(graph, partition)
        modularity_trace.append((len(partition), modularity))

        if modularity > best_modularity:
            best_modularity = modularity
            best_partition = partition

        if len(partition) == graph.number_of_nodes():
            break

    return best_partition, best_modularity, modularity_trace


gn_partition, gn_modularity, gn_trace = best_girvan_newman_partition(G_week6_lcc)
greedy_partition = list(nx.community.greedy_modularity_communities(G_week6_lcc))
greedy_modularity = nx.community.modularity(G_week6_lcc, greedy_partition)

community_comparison = pd.DataFrame(
    {
        'method': ['Girvan-Newman (best modularity partition)', 'Greedy modularity optimization'],
        'num_communities': [len(gn_partition), len(greedy_partition)],
        'modularity': [gn_modularity, greedy_modularity],
        'community_sizes': [
            sorted([len(c) for c in gn_partition], reverse=True),
            sorted([len(c) for c in greedy_partition], reverse=True),
        ],
    }
)

community_comparison

In [ ]:
gn_trace_df = pd.DataFrame(gn_trace, columns=['num_communities', 'modularity'])

plt.figure(figsize=(8, 5))
plt.plot(gn_trace_df['num_communities'], gn_trace_df['modularity'], marker='o')
plt.axhline(greedy_modularity, color='firebrick', linestyle='--', label='Greedy modularity')
plt.xlabel('Number of communities in Girvan-Newman partition')
plt.ylabel('Modularity')
plt.title('Community detection comparison')
plt.legend()
plt.show()

In [ ]:
if greedy_modularity >= gn_modularity:
    best_partition_name = 'Greedy modularity optimization'
    best_partition = greedy_partition
else:
    best_partition_name = 'Girvan-Newman'
    best_partition = gn_partition

partition_map = {}
for community_id, community_nodes in enumerate(best_partition):
    for node in community_nodes:
        partition_map[node] = community_id

preview_layout = nx.spring_layout(G_week6_lcc, seed=42)
node_colors = [partition_map[node] for node in G_week6_lcc.nodes()]

plt.figure(figsize=(12, 9))
nx.draw_networkx_edges(G_week6_lcc, pos=preview_layout, alpha=0.25, edge_color='gray')
nx.draw_networkx_nodes(
    G_week6_lcc,
    pos=preview_layout,
    node_color=node_colors,
    cmap=plt.cm.Set3,
    node_size=220,
)
nx.draw_networkx_labels(
    G_week6_lcc,
    pos=preview_layout,
    labels={node: G_week6_lcc.nodes[node]['label'] for node in G_week6_lcc.nodes()},
    font_size=7,
)
plt.title(f'Best partition preview: {best_partition_name}')
plt.axis('off')
plt.show()

In [ ]:
gexf_graph = G_week6_lcc.copy()
nx.set_node_attributes(gexf_graph, partition_map, 'community')

GEXF_OUTPUT = Path.cwd() / 'catch_me_if_you_can_best_partition.gexf'
nx.write_gexf(gexf_graph, GEXF_OUTPUT)

print(f'Best method: {best_partition_name}')
print(f'Gephi file exported to: {GEXF_OUTPUT}')

### Comment

On this network, the best method is the one with the higher modularity because it produces a partition with stronger within-community cohesion and weaker between-community connectivity. In practice, greedy modularity optimization is often preferable here because it directly searches for a high-modularity partition and is more stable for a graph of this size, while Girvan-Newman can over-split the network as edges are progressively removed.

The exported `.gexf` file can be opened in **Gephi Lite**. Once opened, you can color nodes by the `community` attribute and choose a force-directed layout to obtain the final visualization required by the assignment.

## Week 7

For link prediction, we again treat the graph as **undirected and unweighted**, remove self-loops, and work on the **largest connected component**.

We implement:

1. **Common Neighbors (CN)** as a custom function.
2. **Adamic-Adar (AA)** as the second topological index.
3. A third combined score obtained after rescaling both indices to the `[0, 1]` range and then taking their arithmetic mean.

The output is a pandas DataFrame where each row corresponds to a missing link.

In [ ]:
G_week7 = nx.Graph()
G_week7.add_nodes_from(G.nodes(data=True))
G_week7.add_edges_from((u, v) for u, v in G.edges() if u != v)

if nx.is_connected(G_week7):
    G_week7_lcc = G_week7.copy()
else:
    week7_component_nodes = max(nx.connected_components(G_week7), key=len)
    G_week7_lcc = G_week7.subgraph(week7_component_nodes).copy()

print(
    f"Week 7 graph: {G_week7_lcc.number_of_nodes()} nodes, "
    f"{G_week7_lcc.number_of_edges()} edges"
)

In [ ]:
def common_neighbors_dataframe(graph):
    rows = []
    for u in graph.nodes():
        for v in graph.nodes():
            if u >= v or graph.has_edge(u, v):
                continue
            cn_value = len(list(nx.common_neighbors(graph, u, v)))
            rows.append({'node_u': u, 'node_v': v, 'CN': cn_value})
    return pd.DataFrame(rows)


def min_max_rescale(series):
    minimum = series.min()
    maximum = series.max()
    if maximum == minimum:
        return pd.Series(np.zeros(len(series)), index=series.index)
    return (series - minimum) / (maximum - minimum)


missing_links_df = common_neighbors_dataframe(G_week7_lcc)

adamic_adar_scores = pd.DataFrame(
    nx.adamic_adar_index(G_week7_lcc),
    columns=['node_u', 'node_v', 'AA']
)

link_prediction_df = missing_links_df.merge(adamic_adar_scores, on=['node_u', 'node_v'], how='left')
link_prediction_df['label_u'] = link_prediction_df['node_u'].map(lambda node: G_week7_lcc.nodes[node]['label'])
link_prediction_df['label_v'] = link_prediction_df['node_v'].map(lambda node: G_week7_lcc.nodes[node]['label'])

link_prediction_df['CN_rescaled'] = min_max_rescale(link_prediction_df['CN'])
link_prediction_df['AA_rescaled'] = min_max_rescale(link_prediction_df['AA'])
link_prediction_df['combined_score'] = (link_prediction_df['CN_rescaled'] + link_prediction_df['AA_rescaled']) / 2

link_prediction_df = link_prediction_df[
    ['node_u', 'label_u', 'node_v', 'label_v', 'CN', 'AA', 'CN_rescaled', 'AA_rescaled', 'combined_score']
]

link_prediction_df.head()

In [ ]:
TOP_K = 10

top_cn = link_prediction_df.sort_values(['CN', 'AA'], ascending=False).head(TOP_K).copy()
top_aa = link_prediction_df.sort_values(['AA', 'CN'], ascending=False).head(TOP_K).copy()
top_combined = link_prediction_df.sort_values(['combined_score', 'CN', 'AA'], ascending=False).head(TOP_K).copy()

print('Top missing links by Common Neighbors')
display(top_cn[['label_u', 'label_v', 'CN', 'AA', 'combined_score']])

print('Top missing links by Adamic-Adar')
display(top_aa[['label_u', 'label_v', 'CN', 'AA', 'combined_score']])

print('Top missing links by combined score')
display(top_combined[['label_u', 'label_v', 'CN', 'AA', 'combined_score']])

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(link_prediction_df['CN'], bins=15, color='darkorange', edgecolor='black')
axes[0].set_title('Distribution of CN scores')
axes[0].set_xlabel('CN')
axes[0].set_ylabel('Frequency')

axes[1].hist(link_prediction_df['AA'], bins=15, color='seagreen', edgecolor='black')
axes[1].set_title('Distribution of AA scores')
axes[1].set_xlabel('AA')
axes[1].set_ylabel('Frequency')

axes[2].hist(link_prediction_df['combined_score'], bins=15, color='slateblue', edgecolor='black')
axes[2].set_title('Distribution of combined scores')
axes[2].set_xlabel('Combined score')
axes[2].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

### Comment

Common Neighbors rewards node pairs that already share many mutual contacts, so it tends to favor locally dense areas. Adamic-Adar is more selective because it gives more importance to rare common neighbors and discounts very popular ones. The combined score is useful because it balances the raw number of shared neighbors with the informational value of those neighbors.

If the same pairs appear near the top of all three rankings, the prediction is relatively robust. If the rankings differ, it means that some candidate links are supported by many common neighbors, while others are supported by fewer but more informative common neighbors.

## Short conclusion for Weeks 1-7

Weeks 1-3 describe the structure of the movie network and its clustering patterns. Week 5 identifies the most central characters in terms of brokerage and prestige. Week 6 compares community detection methods and exports the best partition for Gephi. Week 7 studies plausible missing links by combining local topological similarity measures on the unweighted largest connected component.